In [1]:
from multitudcsd.config import get_spark_session
from multitudcsd.storage import read_delta, get_table_path

In [3]:
spark = get_spark_session()

In [4]:
print(spark.sparkContext._jvm.org.apache.hadoop.util.NativeCodeLoader.isNativeCodeLoaded())

True


In [5]:
a = get_table_path("bronze","bronze_gtfs_tripupdates")
print(a)

D:/05_MasterUCM/TFM/multitudcsd/data/lakehouse/bronze/bronze_gtfs_tripupdates


In [6]:
import os
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("hadoop:", spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())
print("nativa cargada:", spark.sparkContext._jvm.org.apache.hadoop.util.NativeCodeLoader.isNativeCodeLoaded())

HADOOP_HOME: C:\Hadoop
hadoop: 3.3.4
nativa cargada: True


In [7]:
from pathlib import Path
print(sorted(p.name for p in Path(r"C:\Hadoop\bin").iterdir()))

jvm = spark.sparkContext._jvm
print(jvm.java.lang.System.getProperty("java.library.path"))

# Esto lanza el error real, que NativeCodeLoader se traga en un LOG.debug.
jvm.java.lang.System.loadLibrary("hadoop")

['hadoop.dll', 'winutils.exe']
C:\Program Files\Java\jdk-17\bin;C:\WINDOWS\Sun\Java\bin;C:\WINDOWS\system32;C:\WINDOWS;C:\Hadoop\bin;D:\05_MasterUCM\TFM\multitudcsd\.venv\Scripts;C:\Program Files\Common Files\Oracle\Java\javapath;C:\Program Files (x86)\Common Files\Oracle\Java\javapath;C:\Program Files (x86)\Intel\iCLS Client\;C:\Program Files\Intel\iCLS Client\;C:\windows\system32;C:\windows;C:\windows\System32\Wbem;C:\windows\System32\WindowsPowerShell\v1.0\;C:\Program Files (x86)\Intel\Intel(R) Management Engine Components\DAL;C:\Program Files\Intel\Intel(R) Management Engine Components\DAL;C:\Program Files (x86)\Intel\Intel(R) Management Engine Components\IPT;C:\Program Files\Intel\Intel(R) Management Engine Components\IPT;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files\Intel\WiFi\bin\;C:\Program Files\Common Files\Intel\WirelessCo

In [8]:
#comprobando que hay datos en las tablas antes de hacer silver y gold
for tabla in ["bronze_gtfs_tripupdates", "bronze_nextbike_status",
              "bronze_viz_disruptions", "bronze_gtfs_static_stops"]:
    df = read_delta(spark, "bronze", tabla)
    print(tabla, df.count())

bronze_gtfs_tripupdates 12585
bronze_nextbike_status 1048
bronze_viz_disruptions 235
bronze_gtfs_static_stops 3005


In [9]:
read_delta(spark, "bronze", "bronze_nextbike_station_information").show(3, truncate=80)

+----------+--------------------------------------------------------------------------------+------------------------------------------------------------------------------+-----------------+------+--------------------------+-----------+
|station_id|                                                                    payload_json|                                                                    source_url|feed_last_updated|source|                 ingest_ts|ingest_date|
+----------+--------------------------------------------------------------------------------+------------------------------------------------------------------------------+-----------------+------+--------------------------+-----------+
| 116227173|{"station_id": "116227173", "name": "EDEKA Wiesbadener Straße", "short_name":...|https://gbfs.nextbike.net/maps/gbfs/v2/nextbike_bn/en/station_information.json|       1788299239|  real|2026-09-01 23:48:18.800353| 2026-09-01|
| 116227471|{"station_id": "116227471", "name": "EDE

In [10]:
from multitudcsd.transforms.geo import compute_h3_cell

# Puerta de Brandeburgo, punto de referencia del recorrido del CSD
celda = compute_h3_cell(52.5163, 13.3777, resolution=9)
print(celda)

891f1d48863ffff


In [11]:
bronze_info.select("payload_json").show(1, truncate=False)

NameError: name 'bronze_info' is not defined

In [15]:
def _inspecciona_un_payload_de_viz(spark):
    from multitudcsd.storage import read_delta
    df = read_delta(spark, "bronze", "bronze_viz_disruptions")
    df.select("payload_json").show(5, truncate=False)

In [16]:
print(_inspecciona_un_payload_de_viz(spark))

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
silver_bikes = read_delta(spark, "silver", "silver_bike_availability")
silver_bikes.select("station_id", "h3_index", "num_bikes_available", "reading_ts").show(5)

silver_delays = read_delta(spark, "silver", "silver_transit_delays")
silver_delays.select("route_id", "stop_id", "h3_index", "delay_seconds", "feed_ts").show(5)

silver_disruptions = read_delta(spark, "silver", "silver_disruptions")
silver_disruptions.select(
    "disruption_id", "h3_index", "street", "severity", "valid_from", "valid_to"
).show(5, truncate=40)

In [17]:
silver_bikes = read_delta(spark, "silver", "silver_bike_availability")
silver_bikes.select("station_id", "h3_index", "num_bikes_available", "reading_ts").show(5)


+----------+---------------+-------------------+-------------------+
|station_id|       h3_index|num_bikes_available|         reading_ts|
+----------+---------------+-------------------+-------------------+
| 167312062|891f1d48e9bffff|                  3|2026-08-23 23:15:00|
| 167339040|891f1d4d063ffff|                  1|2026-08-23 23:15:00|
| 167369847|891f1d4d10bffff|                  2|2026-08-23 23:15:00|
| 167375258|891f1d4d1cbffff|                  0|2026-08-23 23:15:00|
| 167424125|891f1d4f65bffff|                  1|2026-08-23 23:15:00|
+----------+---------------+-------------------+-------------------+
only showing top 5 rows



In [26]:
silver_delays = read_delta(spark, "silver", "silver_transit_delays")
silver_delays.select("route_id", "stop_id", "h3_index", "delay_seconds", "feed_ts").show(50)


+---------+--------------------+---------------+-------------+-------------------+
| route_id|             stop_id|       h3_index|delay_seconds|            feed_ts|
+---------+--------------------+---------------+-------------+-------------------+
| 1928_700|de:12063:90021000...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021020...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021021...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021024...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021032...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021050...|           NULL|            0|2026-08-23 23:53:50|
| 1928_700|de:12063:90021050...|           NULL|            0|2026-08-23 23:53:50|
| 19

In [27]:
silver_disruptions = read_delta(spark, "silver", "silver_disruptions")
silver_disruptions.select(
    "disruption_id", "h3_index", "street", "severity", "valid_from", "valid_to"
).show(5, truncate=40)

+-------------+---------------+------+--------+----------+--------+
|disruption_id|       h3_index|street|severity|valid_from|valid_to|
+-------------+---------------+------+--------+----------+--------+
|             |891f1d4ab2bffff|  NULL|    NULL|      NULL|    NULL|
+-------------+---------------+------+--------+----------+--------+



In [ ]:
gold_mp = read_delta(spark, "gold", "gold_mobility_pressure")
gold_mp.orderBy(F.desc("num_lecturas_bici")).show(10)

gold_reliability = read_delta(spark, "gold", "gold_line_reliability")
gold_reliability.orderBy(F.desc("avg_delay_seconds")).show(10)

gold_disruptions = read_delta(spark, "gold", "gold_disruptions_by_cell")
gold_disruptions.orderBy(F.desc("num_disruptions")).show(10)

In [22]:
from pyspark.sql import functions as F
gold_mp = read_delta(spark, "gold", "gold_mobility_pressure")
gold_mp.orderBy(F.desc("num_lecturas_bici")).show(10)

+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|       h3_index|hour_of_day|avg_bikes_available|avg_docks_available|num_lecturas_bici| avg_delay_seconds|       pct_on_time|num_actualizaciones_retraso|
+---------------+-----------+-------------------+-------------------+-----------------+------------------+------------------+---------------------------+
|891f1d48b13ffff|         23| 0.3333333333333333|  5.833333333333333|               12|               7.5|            0.9375|                         16|
|891f1d48c13ffff|         23| 0.8333333333333334| 3.1666666666666665|               12|              NULL|              NULL|                       NULL|
|891f1d48903ffff|         23|                2.9|                4.4|               10|              NULL|              NULL|                       NULL|
|891f1d49b77ffff|         23|                1.7|                3.4|       

In [28]:
gold_reliability = read_delta(spark, "gold", "gold_line_reliability")
gold_reliability.orderBy(F.desc("avg_delay_seconds")).show(10)


+---------+-----------+------------------+-------------------+-------------------+
| route_id|hour_of_day| avg_delay_seconds|        pct_on_time|num_actualizaciones|
+---------+-----------+------------------+-------------------+-------------------+
|20969_700|          0|2612.3333333333335|0.16666666666666666|                  6|
|17435_700|          0|1000.6451612903226| 0.6451612903225806|                 31|
|28259_100|          0|             912.0|                0.0|                 10|
|26788_106|          0| 652.2631578947369| 0.3684210526315789|                 19|
|17437_700|          0|477.77777777777777| 0.6666666666666666|                 27|
|17398_700|          0|409.09090909090907|                0.0|                 11|
|17499_700|          0|             300.0|                0.0|                 12|
|17535_700|         23|230.52631578947367| 0.5263157894736842|                 19|
|17281_700|          0|226.41509433962264| 0.8301886792452831|                 53|
|240

In [29]:
gold_disruptions = read_delta(spark, "gold", "gold_disruptions_by_cell")
gold_disruptions.orderBy(F.desc("num_disruptions")).show(10)

+--------+---------------+
|h3_index|num_disruptions|
+--------+---------------+
+--------+---------------+

